# Face Unmasking & Emotion Recognition Research

This notebook implements a Deep Learning pipeline to:
1.  **Synthesize Face Masks** on standard datasets (FER2013) to create paired training data.
2.  **Train an Unmasking Model (Autoencoder/U-Net)** to remove masks and reconstruct the original face.
3.  **Train an Emotion Recognition Model (CNN)** on the unmasked faces.
4.  **Evaluate on Real-World Data**: Use the provided Masked FER Dataset for validation.

## Dependencies
Ensure you have `tensorflow`, `opencv-python`, `pandas`, `numpy`, `matplotlib`, and `scikit-learn` installed.

In [ ]:
import os
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras import layers, models, Model
from sklearn.model_selection import train_test_split

# Constants
IMG_SIZE = (64, 64)
BATCH_SIZE = 32
fer_path = 'fer2013.csv'
masked_dataset_path = 'dataset/extracted/masked_dataset/train' # Adjust if needed

ModuleNotFoundError: No module named 'cv2'

## 1. Load Data
We load the FER2013 dataset. If you don't have it, ensure `download_fer2013.py` has run.

In [ ]:
def load_fer2013(path, max_samples=10000):
    print(f"Loading FER2013 from {path}...")
    df = pd.read_csv(path)
    if max_samples:
        df = df.head(max_samples)
    
    pixels = df['pixels'].tolist()
    emotions = pd.get_dummies(df['emotion']).values
    
    X = []
    for p in pixels:
        face = [int(pixel) for pixel in p.split(' ')]
        face = np.asarray(face).reshape(48, 48)
        face = cv2.resize(face.astype('uint8'), IMG_SIZE)
        X.append(face.astype('float32') / 255.0)
        
    return np.array(X)[..., np.newaxis], emotions

if os.path.exists(fer_path):
    X_faces, y_emotions = load_fer2013(fer_path)
    print(f"Loaded {len(X_faces)} images from FER2013.")
else:
    print("fer2013.csv not found! Please run download_fer2013.py first.")
    # Creating dummy data for demonstration if file missing
    X_faces = np.random.rand(100, 64, 64, 1).astype('float32')
    y_emotions = np.random.randint(0, 2, (100, 7))

## 2. Synthetic Mask Generation (The "Mask R-CNN" Concept)
To train our Unmasker, we need GROUND TRUTH. 
We take clean FER2013 images and synthetically apply masks. This gives us perfect pairs: `(Masked Image, Original Image)`.

**Note:** Real Mask R-CNN isolates objects. Here, we simulate that step by creating the mask overlay ourselves.

In [ ]:
def apply_mask(image):
    masked = image.copy()
    h, w, _ = masked.shape
    # Simple rectangular mask (simulating surgical mask position)
    # In a real paper, use 'MaskTheFace' for realistic overlays
    cv2.rectangle(masked, (0, h//2), (w, h), (0, 0, 0), -1) 
    return masked

X_masked = np.array([apply_mask(x) for x in X_faces])

# Visualize
plt.figure(figsize=(10, 4))
for i in range(5):
    ax = plt.subplot(2, 5, i + 1)
    plt.imshow(X_faces[i].squeeze(), cmap='gray')
    plt.title("Original")
    plt.axis("off")
    
    ax = plt.subplot(2, 5, i + 1 + 5)
    plt.imshow(X_masked[i].squeeze(), cmap='gray')
    plt.title("Masked")
    plt.axis("off")
plt.show()

## 3. Build & Train Unmasking Model (Autoencoder)
This model learns to reconstruct the original face from the masked input.

In [ ]:
def build_unet_autoencoder():
    inputs = layers.Input(shape=(64, 64, 1))
    
    # Encoder
    c1 = layers.Conv2D(32, (3, 3), activation='relu', padding='same')(inputs)
    p1 = layers.MaxPooling2D((2, 2))(c1)
    c2 = layers.Conv2D(64, (3, 3), activation='relu', padding='same')(p1)
    p2 = layers.MaxPooling2D((2, 2))(c2)
    
    # Bottleneck
    b = layers.Conv2D(128, (3, 3), activation='relu', padding='same')(p2)
    
    # Decoder
    u1 = layers.UpSampling2D((2, 2))(b)
    concat1 = layers.Concatenate()([u1, c2])
    c3 = layers.Conv2D(64, (3, 3), activation='relu', padding='same')(concat1)
    
    u2 = layers.UpSampling2D((2, 2))(c3)
    concat2 = layers.Concatenate()([u2, c1])
    c4 = layers.Conv2D(32, (3, 3), activation='relu', padding='same')(concat2)
    
    outputs = layers.Conv2D(1, (3, 3), activation='sigmoid', padding='same')(c4)
    
    return Model(inputs, outputs)

unmasker = build_unet_autoencoder()
unmasker.compile(optimizer='adam', loss='mse')
unmasker.summary()

In [ ]:
# Train Unmasker
history_unmask = unmasker.fit(
    X_masked, X_faces,
    epochs=10,
    batch_size=32,
    validation_split=0.2
)

## 4. Build & Train Emotion Classifier
We train this on the *clean origin* FER2013 images to establish ground truth emotion recognition.

In [ ]:
def build_classifier(num_classes):
    inputs = layers.Input(shape=(64, 64, 1))
    x = layers.Conv2D(32, (3, 3), activation='relu')(inputs)
    x = layers.MaxPooling2D((2, 2))(x)
    x = layers.Conv2D(64, (3, 3), activation='relu')(x)
    x = layers.MaxPooling2D((2, 2))(x)
    x = layers.Flatten()(x)
    x = layers.Dense(64, activation='relu')(x)
    outputs = layers.Dense(num_classes, activation='softmax')(x)
    return Model(inputs, outputs)

classifier = build_classifier(y_emotions.shape[1])
classifier.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

# Train Classifier
history_classify = classifier.fit(
    X_faces, y_emotions,
    epochs=10,
    batch_size=32,
    validation_split=0.2
)

## 5. Evaluation on Real World Data
Now we test the pipeline on your downloaded dataset (Real Masked Faces).
1. Load a real masked image.
2. Pass it through Unmasker.
3. Pass reconstruction to Classifier.

In [ ]:
def predict_pipeline(image_path):
    # Load Image
    img = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)
    if img is None:
        print(f"Could not load {image_path}")
        return
    img_resized = cv2.resize(img, IMG_SIZE).astype('float32') / 255.0
    img_input = img_resized[np.newaxis, ..., np.newaxis]
    
    # Unmask
    reconstructed = unmasker.predict(img_input)
    
    # Classify
    emotion_probs = classifier.predict(reconstructed)
    emotion_idx = np.argmax(emotion_probs)
    
    # Visualize
    plt.figure(figsize=(10, 4))
    ax = plt.subplot(1, 3, 1)
    plt.imshow(img_resized, cmap='gray')
    plt.title("Input (Masked)")
    plt.axis('off')
    
    ax = plt.subplot(1, 3, 2)
    plt.imshow(reconstructed.squeeze(), cmap='gray')
    plt.title("Unmasked (Output)")
    plt.axis('off')
    
    ax = plt.subplot(1, 3, 3)
    plt.bar(range(len(emotion_probs[0])), emotion_probs[0])
    plt.title(f"Emotion Prediction: {emotion_idx}")
    plt.show()

# Test on a few images from your dataset
if os.path.exists(masked_dataset_path):
    print("Testing on Real Dataset files...")
    # Walk through folders like 'angry', 'fear' etc
    classes = os.listdir(masked_dataset_path)
    for class_folder in classes[:3]: # Check first 3 classes
        full_path_dir = os.path.join(masked_dataset_path, class_folder)
        if os.path.isdir(full_path_dir):
             # Get one image
             subfiles = os.listdir(full_path_dir)
             if subfiles:
                 img_path = os.path.join(full_path_dir, subfiles[0])
                 print(f"Testing Class: {class_folder}")
                 predict_pipeline(img_path)
else:
    print(f"Path {masked_dataset_path} not found. Testing on synthetic validation set.")
    # Test on synthetic validation set
    test_idx = 0
    test_img = X_masked[test_idx]
    img_input = test_img[np.newaxis, ...]
    reconstructed = unmasker.predict(img_input)
    emotion_probs = classifier.predict(reconstructed)
    
    plt.figure(figsize=(10, 4))
    plt.subplot(1, 2, 1)
    plt.imshow(test_img.squeeze(), cmap='gray')
    plt.title("Input Synthetic Masked")
    plt.subplot(1, 2, 2)
    plt.imshow(reconstructed.squeeze(), cmap='gray')
    plt.title("Reconstructed")
    plt.show()